🔹 Step 1: Environment Setup

In [141]:
# Install required libraries (run once)
# pip install pandas numpy matplotlib scikit-learn tensorflow

In [142]:
!python --version

Python 3.12.12


🔹 Step 2: Import Libraries

In [143]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input, Bidirectional
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping

🔹 Step 3: Load & Inspect Dataset

In [144]:
df = pd.read_csv("Job_3_Resource_sentiment.csv")
print(df.columns)

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')


In [145]:
df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [146]:
df.columns

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')

In [147]:
df.rename(columns={'Positive': 'sentiment'}, inplace=True)
df.rename(columns={'im getting on borderlands and i will murder you all ,': 'text'}, inplace=True)

In [148]:
df.columns

Index(['2401', 'Borderlands', 'sentiment', 'text'], dtype='object')

In [149]:
df.sample(5)

,2401,Borderlands,sentiment,text
58608,3248,Facebook,Positive,I love how it sounds like a hot new TV show.
20137,12647,WorldOfCraft,Positive,Wonderful views of Dreyfus...
54709,2196,CallOfDuty,Irrelevant,Tch.tv / ace _ crussty. @ ScrimFinder. @ FearF...
53439,10781,RedDeadRedemption(RDR),Neutral,R i am arthur morgan from red dead redemption ...
5890,214,Amazon,Neutral,The exterior of the bell USES electroplating p...


In [150]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   2401         74681 non-null  int64 
 1   Borderlands  74681 non-null  object
 2   sentiment    74681 non-null  object
 3   text         73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [151]:
df = df[['text', 'sentiment']]

In [152]:
df.sample(5)

,text,sentiment
60584,Sorry for recent tech difficulties at @Phoenix...,Neutral
28788,Nominated by (because it annoyed me). @ Kat3pi...,Neutral
3988,Yikes. you have to pay more to play the both c...,Negative
42419,Even tae playing pubg I won’t ever installing ...,Negative
72297,And Beautiful.,Positive


In [153]:
df

,text,sentiment
0,I am coming to the borders and I will kill you...,Positive
1,im getting on borderlands and i will kill you ...,Positive
2,im coming on borderlands and i will murder you...,Positive
3,im getting on borderlands 2 and i will murder ...,Positive
4,im getting into borderlands and i can murder y...,Positive
...,...,...
74676,Just realized that the Windows partition of my...,Positive
74677,Just realized that my Mac window partition is ...,Positive
74678,Just realized the windows partition of my Mac ...,Positive
74679,Just realized between the windows partition of...,Positive


In [154]:
df.shape

(74681, 2)

In [155]:
df.isnull().sum()

,0
text,686
sentiment,0


In [156]:
print(df['sentiment'].value_counts())

sentiment
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


In [157]:
df.duplicated().sum()

np.int64(4909)

🔹 Step 4: Data Cleaning

In [158]:
df.dropna(inplace=True)

In [159]:
df.isnull().sum()

,0
text,0
sentiment,0


In [160]:
df['text'] = df['text'].astype(str)

In [161]:
df['text']

,text
0,I am coming to the borders and I will kill you...
1,im getting on borderlands and i will kill you ...
2,im coming on borderlands and i will murder you...
3,im getting on borderlands 2 and i will murder ...
4,im getting into borderlands and i can murder y...
...,...
74676,Just realized that the Windows partition of my...
74677,Just realized that my Mac window partition is ...
74678,Just realized the windows partition of my Mac ...
74679,Just realized between the windows partition of...


In [162]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text.strip()

df['text'] = df['text'].apply(clean_text)

In [163]:
df['text']

,text
0,i am coming to the borders and i will kill you...
1,im getting on borderlands and i will kill you all
2,im coming on borderlands and i will murder you...
3,im getting on borderlands and i will murder y...
4,im getting into borderlands and i can murder y...
...,...
74676,just realized that the windows partition of my...
74677,just realized that my mac window partition is ...
74678,just realized the windows partition of my mac ...
74679,just realized between the windows partition of...
